# Predicting Player Contributions to PLAICraft with Experience and Age

## Introduction

Embodied artificial intelligence (AI) is AI that can think, act, and learn in real-world and digital in simulated environments. Such a program would be able to act like a real person in environments such as video games.

PLAICraft is a research data collection project that uses the video game "Minecraft" to work towards creating embodied AI. Using its own Minecraft server, they collect detailed information on how participants think and move while playing, as well as their demographic information such as age, experience, and gender.

For the PLAICraft Minecraft server, the more hours the players play, the more data is collected.

Research data collection projects like PLAICraft need large amounts of data to be able to provide meaningful results, and so the team need to be able to recruit participants that are likely to contribute a large amount of data. With that goal in mind, we investigated one of the questions posed by the PLAICraft research team, which was "which 'kinds' of players are most likely to contribute a large amount of data?"

Specifically, based on the demographic data available to us, our question became **can the Age and experience of a player be used to predict the sum total of hours played across multiple sessions of PLAICraft?**

Since the research team is looking for 'kinds' of players, we include Age as demographic information. We decided against including gender because the data is lacking in many observations for most non-male genders, and so we don't have sufficient training data to use it as a predictor. experience is the one additional detail for 'type' of player, and we convert it from 5 discrete factors into a linear scale of 1-5 in the order Amateur, Beginner, Regular, Veteran, and Pro. We use a linear scale instead of categories so we can take advantage of that fact that an Amateur and Beginner are a more similar 'type' of player than an Amateur and a Pro, a relationship that our model couldn't take into account if we treated experience as a factor.

The researchers are asking to identify predictors for players who contribute a "large amount" of data which could be considered categorical, but we plan to treat it as a regression problem where we predict the numerical value of played_hours for a given type of player. This gives the researchers the ability to define their own cutoff for what minimum number of hours is considered 'large'.

In [1]:
library(tidyverse)
library(lubridate)

players_data = download.file("https://github.com/apepers/DSCI_100_009_33/raw/refs/heads/main/players.csv", "players.csv")
players = read_csv("players.csv")

NameError: name 'library' is not defined

## Methods & Results

- describe the methods you used to perform your analysis from beginning to end that narrates the analysis code.
- your report should include code which:
    - loads data
    - wrangles and cleans the data to the format necessary for the planned analysis
    - performs a summary of the data set that is relevant for exploratory data analysis related to the planned analysis
    - creates a visualization of the dataset that is relevant for exploratory data analysis related to the planned analysis
    - performs the data analysis
    - creates a visualization of the analysis
    - note: all figures should have a figure number and a legend

Our first step was to load, tidy, and perform exploratory visualizations with the dataset provided by the PLAICraft researchers.

In [6]:
library(tidyverse)
library(lubridate)

# Load dataset
players_data = download.file("https://github.com/apepers/DSCI_100_009_33/raw/refs/heads/main/players.csv", "players.csv")
players = read_csv("players.csv")
head(players)

Rows: 196 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): experience, hashedEmail, name, gender
dbl (2): played_hours, Age
lgl (1): subscribe

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


#### Summary of Dataset
`players.csv` includes information about each player, with 7 variables:

|Variable|Type|Description|
|-----------|-------|------------|
|experience|character|The player's experience level|
|subscribe|logical|If the player is subscribed to a game-related newsletter|
|hashedEmail|character|Unique hashed email of player|
|played_hours|double|Hours player has played|
|name|character|First name of player|
|gender|character|Gender of player|
|Age|double|Age of player (years)|

In [ ]:
obsPlayers <- summarize(players, count = n()) |> pull(count)
obsSessions <- summarize(sessions, count = n()) |> pull(count)
print(paste("Number of Observations in Players:", obsPlayers))
print(paste("Number of Observations in Sessions:", obsSessions))

In [ ]:
summarizedPlayers <- summary(players)
summarizedPlayers

experience convert to a 1-5 scale.

In [2]:
players<-mutate(players, experience_level = as.numeric(factor(experience, levels = c("Beginner", "Regular", "Amateur", "Veteran", "Pro"))))
head(players)

SyntaxError: invalid syntax (3952348532.py, line 1)

### Exploratory Data Analysis

#### Age

In [3]:
# First explore the Age and hours played relationship with a scatterplot
age_time_plot = players |>
    ggplot(aes(x = Age, y = played_hours)) +
    geom_point(na.rm = TRUE) +
    labs(x = "Age (years)", y = "Total Hours Played") +
    ggtitle("Total Hours Played vs Age (0 duration included)")

age_time_plot

SyntaxError: invalid syntax (3750722268.py, line 2)

As we'll discuss in more detail later, the player data includes many players with 0 hours played, as well as a few outliers with over 100 hours played. To get a better view of the relationship between Age and played_hours among players who did play, we filter out the players with no hours, and switch to a logarithmic scale.

In [ ]:
age_time_plot_no_0 = players |>
    filter(played_hours > 0) |>
    ggplot(aes(x = Age, y = played_hours)) +
    geom_point(na.rm = TRUE) +
    labs(x = "Age (years)", y = "Total Hours Played (> 0)") +
    ggtitle("Total Hours Played vs Age (0 Duration excluded, logarithmic scaled)") +
    scale_y_log10()

age_time_plot_no_0

Experience level

In [ ]:
Distribuition_plot_0<-ggplot(players, aes(x = experience, y = played_hours, fill = experience)) +
  geom_boxplot() +
  labs(title = "Distribution of Time Played Vs Experience Level",
    x = "Experience Level",
    y = "Time Played (Hours)")+
scale_fill_brewer(palette = "Set3")
Distribuition_plot_0

This graph includes all zero-hour playtime and shows the full pattern that many players did not contribute large amounts of data to the server. This is evident in the highly compressed y-axis values due to the large number of zero-hour players.

In [5]:
Distribuition_plot_no_0<-players|>
 filter(played_hours > 0) |>  
ggplot(aes(x = experience, y = played_hours, fill = experience)) +
  geom_boxplot() +
  labs(title = "Distribution of Time Played Vs Experience Level (Only Non-Zero Gameplay)",
    x = "Experience Level",
    y = "Time Played (Hours)") +
 scale_y_log10() + #log10 cannot display 0 values, this helps visualize the non-zero distribuition better
scale_fill_brewer(palette = "Set3")
Distribuition_plot_no_0

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 8)

However, this plot with only non-zero hours helps focus in on the patterns amongst the players who did contribute data. This non-zero graph also allows for meaningful comparison across experience levels and helps identify which player groups will play longer and therefore, contribute more data to the server.

## Discussion

- summarize what you found
- discuss whether this is what you expected to find
- discuss what impact could such findings have
- discuss what future questions could this lead to


Summarize what our found: The game is mainly popular among younger age groups. People aged between 10 and 30 tend to play it for a longer time and there are also more of them.

Discuss whether this is what youe expected to find: These comparisons are in line with what we had expected-young people are more willing to spend more time on games than the elderly.

Discuss what impact could such findings have: derstanding the gaming behavior of players of different age groups can provide valuable insights for game developers and marketing teams. For example, it can help them design game mechanics that better suit the preferences of younger players. Developers can also improve the gaming experience for older players by adjusting the game's difficulty, which a way to increase playtime and player numbers. Identifying which player groups contribute the most playtime allows marketing teams to develop more effective marketing strategies.

Discuss what future questions could this lead to: Besides age and experience, what other factors can we use to predict which type of player can provide more data to the company?

## References

- You may include references if necessary, as long as they all have a consistent citation style.
